# Import Library

In [1]:
import os
import warnings
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

# 1. โหลดข้อมูลหลัก
print("Loading application dataset...")
df_train = pd.read_csv("../data/raw/application_train.csv")
print(f"Base Train Shape: {df_train.shape}")

Loading application dataset...
Base Train Shape: (307511, 122)


# Financial Domain Ratios

* นำตัวแปรทางการเงินหลายตัวมาหารหรือรวมกันตามหลักการวิเคราะห์สินเชื่อของสถาบันการเงิน (Credit Underwriting) เพื่อสะท้อน "พฤติกรรมและความสามารถในการชำระหนี้ที่แท้จริง" ซึ่งมักมีพลังในการทำนาย (Predictive Power) สูงกว่าการดูตัวเลขเดี่ยว

In [2]:
# 1. สร้างกลุ่มฟีเจอร์อัตราส่วนทางการเงิน (Domain Ratios)
df_domain = df_train.copy()

# จัดการ Anomaly ของ DAYS_EMPLOYED ก่อนคำนวณ
df_domain['DAYS_EMPLOYED_ANOM'] = df_domain['DAYS_EMPLOYED'] == 365243
df_domain['DAYS_EMPLOYED'] = df_domain['DAYS_EMPLOYED'].replace({365243: np.nan})

# DTI & Repayment Ratios
df_domain['CREDIT_INCOME_PERCENT'] = df_domain['AMT_CREDIT'] / df_domain['AMT_INCOME_TOTAL']
df_domain['ANNUITY_INCOME_PERCENT'] = df_domain['AMT_ANNUITY'] / df_domain['AMT_INCOME_TOTAL']
df_domain['CREDIT_TERM'] = df_domain['AMT_ANNUITY'] / df_domain['AMT_CREDIT']
df_domain['DAYS_EMPLOYED_PERCENT'] = df_domain['DAYS_EMPLOYED'] / df_domain['DAYS_BIRTH']

# External Sources Aggregations (รวมพลัง External Scores)
df_domain['EXT_SOURCES_MEAN'] = df_domain[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
df_domain['EXT_SOURCES_STD'] = df_domain[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].std(axis=1)
df_domain['EXT_SOURCES_PROD'] = df_domain['EXT_SOURCE_1'] * df_domain['EXT_SOURCE_2'] * df_domain['EXT_SOURCE_3']

# ตรวจสอบ Correlation ของฟีเจอร์ใหม่เทียบกับ TARGET
new_domain_cols = [
    'CREDIT_INCOME_PERCENT', 'ANNUITY_INCOME_PERCENT', 'CREDIT_TERM',
    'DAYS_EMPLOYED_PERCENT', 'EXT_SOURCES_MEAN', 'EXT_SOURCES_STD', 'EXT_SOURCES_PROD'
]

domain_corr = df_domain[new_domain_cols + ['TARGET']].corr()['TARGET'].sort_values()
print("Correlation of New Domain Features with TARGET:")
print(domain_corr)

Correlation of New Domain Features with TARGET:
EXT_SOURCES_MEAN         -0.222052
EXT_SOURCES_PROD         -0.188552
DAYS_EMPLOYED_PERCENT    -0.067955
CREDIT_INCOME_PERCENT    -0.007727
CREDIT_TERM               0.012704
ANNUITY_INCOME_PERCENT    0.014265
EXT_SOURCES_STD           0.047700
TARGET                    1.000000
Name: TARGET, dtype: float64


### 📐 Financial Domain Ratios & Interaction Evaluation

#### 1. ผลลัพธ์เชิงสถิติและการเพิ่มขึ้นของ Predictive Power
* **Strongest Signal Enhancement:**
  * `EXT_SOURCES_MEAN` ได้ค่า Pearson Correlation สูงถึง **-0.222** ซึ่งมี Predictive Signal สูงกว่าตัวแปรเดี่ยวเดิม (`EXT_SOURCE_3` ที่ -0.179) อย่างมีนัยสำคัญ
  * การสร้าง Composite Score ช่วยลดความผันผวนเฉพาะแหล่ง (Idiosyncratic Variance) และเสริมความแม่นยำในการคัดแยกลูกหนี้
* **Job Stability Proxy:**
  * `DAYS_EMPLOYED_PERCENT` (**-0.068**) พิสูจน์ให้เห็นว่าอัตราส่วนความต่อเนื่องในการทำงานต่ออายุตัว เป็นตัวแปรเชิงเสถียรภาพทางการเงินที่มีประสิทธิภาพ
* **Score Uncertainty Metric:**
  * `EXT_SOURCES_STD` (**+0.048**) สะท้อนความไม่สอดคล้องกันของการประเมินเครดิต ซึ่งเชื่อมโยงโดยตรงกับความเสี่ยงการผิดนัดชำระที่สูงขึ้น

---

#### 2. Domain Features Summary Table

| Feature Name | Pearson Corr with TARGET | Business Rationale & Signal Type |
| :--- | :---: | :--- |
| **`EXT_SOURCES_MEAN`** | **-0.222** | คะแนนเครดิตเฉลี่ยรวม (Strongest Protective Factor) |
| **`EXT_SOURCES_PROD`** | **-0.189** | ผลคูณคะแนนเครดิต บ่งชี้ความน่าเชื่อถือร่วมทุกมิติ |
| **`DAYS_EMPLOYED_PERCENT`**| **-0.068** | สัดส่วนความมั่นคงในอาชีพการงานต่อช่วงอายุ |
| **`EXT_SOURCES_STD`** | **+0.048** | ความผันผวน/ขัดแย้งของคะแนนเครดิต (Uncertainty Risk) |
| **`ANNUITY_INCOME_PERCENT`**| **+0.014** | สัดส่วนภาระค่างวดต่อรายได้ (Debt Burden Indicator) |
| **`CREDIT_TERM`** | **+0.013** | ความกดดันของอัตราการผ่อนชำระต่อวงเงินรวม |
| **`CREDIT_INCOME_PERCENT`** | **-0.008** | อัตราส่วนวงเงินกู้รวมต่อรายได้ทั้งปี |